<a href="https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_07/03_lineares_system_1d_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1D Lineares Kinematisches System: Zustandsraummodellierung & Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_07/03_lineares_system_1d_simulation.ipynb)

In diesem Notebook modellieren und simulieren wir ein **1-dimensionales lineares dynamisches System** eines sich bewegenden Objekts (hier visualisiert als ein **Fahrrad**).

### Physikalische Zustandsgrößen:
Der Zustand zum diskreten Zeitschritt $t_k$ wird durch den Zustandsvektor $x_k \in \mathbb{R}^3$ beschrieben:

$$
x_k = \begin{bmatrix} p_k \\ v_k \\ a_k \end{bmatrix}
$$

Dabei bezeichnen:
- $p_k$: Position zum Zeitpunkt $t_k$ (in $\text{m}$)
- $v_k$: Geschwindigkeit zum Zeitpunkt $t_k$ (in $\text{m/s}$)
- $a_k$: Beschleunigung zum Zeitpunkt $t_k$ (in $\text{m/s}^2$)

### Zeitdiskretisierung:
Der zeitliche Abstand zwischen zwei aufeinanderfolgenden Zeitschritten $t_k$ und $t_{k+1}$ ist fest vorgegeben:

$$
\Delta t = t_{k+1} - t_k = 1.0\,\text{s}
$$

### Kinematische Bewegungsgleichungen:
Bei einer konstanten Beschleunigung $a_k$ über das Intervall $[t_k, t_{k+1})$ gilt:

$$
p_{k+1} = p_k + v_k \cdot \Delta t + \frac{1}{2} a_k \cdot \Delta t^2
$$

$$
v_{k+1} = v_k + a_k \cdot \Delta t
$$

In vektorieller Matrix-Schreibweise ergibt sich das lineare Zustandsraummodell:

$$
x_{k+1} = A \cdot x_k
$$

mit der Systemmatrix $A \in \mathbb{R}^{3 \times 3}$:

$$
A = \begin{bmatrix}
1 & \Delta t & \frac{1}{2}\Delta t^2 \\
0 & 1 & \Delta t \\
0 & 0 & 1
\end{bmatrix}
$$


## 1. Benötigte Bibliotheken und Skript importieren

Wir importieren `numpy`, `matplotlib.pyplot`, `IPython.display.HTML` sowie das Hilfsmodul `03_lineares_system_helper.py`, welches die Plot- und Animationsroutinen bereitstellt.

In [ ]:
import os
import requests

if not os.path.exists('03_lineares_system_helper.py'):
    url = 'https://raw.githubusercontent.com/dgaida/wpf_cps_th_public/main/assets/exercises/week_07/03_lineares_system_helper.py'
    r = requests.get(url)
    if r.status_code == 200:
        with open('03_lineares_system_helper.py', 'wb') as f:
            f.write(r.content)
        print('Downloaded 03_lineares_system_helper.py')

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

# Pfad zum lokalen Verzeichnis hinzufügen, um das Hilfsskript zu importieren
sys.path.append('.')
sys.path.append('exercises/source code/week_07')
import importlib
helper_module = importlib.import_module('03_lineares_system_helper')

# Inline-Darstellung konfigurieren
%matplotlib inline
plt.rcParams['font.size'] = 11

print("Modul 03_lineares_system_helper erfolgreich importiert!")

## 2. Aufgabe: Programmierung der Systemmatrix $A$

Definieren Sie das Zeitintervall $\Delta t = 1.0\,\text{s}$ und programmieren Sie die Systemmatrix $A \in \mathbb{R}^{3 \times 3}$ als zweidimensionales NumPy-Array (`np.ndarray`).

Nutzen Sie dazu die exakte kinematische Zustandsübergangsmatrix:
$$
A = \begin{bmatrix}
1 & \Delta t & \frac{1}{2}\Delta t^2 \\
0 & 1 & \Delta t \\
0 & 0 & 1
\end{bmatrix}
$$

In [ ]:
# Zeitabstand Delta t zwischen t_{k+1} und t_k in Sekunden
dt = 1.0

# TODO: Programmieren Sie hier die Systemmatrix A als 3x3 NumPy-Array
# A = np.array([
#     [..., ..., ...],
#     [..., ..., ...],
#     [..., ..., ...]
# ], dtype=float)

A = None  # Ersetzen Sie None durch Ihre Matrix-Definition

print("Systemmatrix A:")
print(A)


### Validierung der Systemmatrix $A$

Führen Sie die nachfolgende Zelle aus, um zu prüfen, ob die Matrix $A$ die korrekte Dimension und Werte besitzt.

In [ ]:
if A is None:
    raise ValueError("Bitte definieren Sie die Systemmatrix A in der vorherigen Zelle!")

assert isinstance(A, np.ndarray), "A muss ein NumPy-Array sein!"
assert A.shape == (3, 3), f"A muss Dimension (3, 3) haben, hat aber {A.shape}."

A_expected = np.array([[1.0, dt, 0.5 * dt**2], [0.0, 1.0, dt], [0.0, 0.0, 1.0]])
np.testing.assert_allclose(A, A_expected, err_msg="Werte der Systemmatrix A stimmen nicht mit den theoretischen Kinematikgleichungen überein!")
print("✓ Sehr gut! Die Systemmatrix A ist korrekt implementiert.")


## 3. Vorgabe der Beschleunigung $a_k$ als stückweise diskrete Funktion

Die Studierenden können die Beschleunigung als stückweise diskrete Funktion vorgeben. Bspw. als Vektor, wobei jeder Wert einer Beschleunigung zum diskreten Zeitschritt $t_k$ entspricht:

$$
a = [a_0, a_1, a_2, \dots, a_{N-1}]
$$

Ein typisches Fahrszenario für ein Fahrrad:
1. **Anfahren**: Positive Beschleunigung ($a_k > 0$)
2. **Gleichförmige Fahrt**: Geschwindigkeit wird gehalten ($a_k = 0$)
3. **Bremsen**: Negative Beschleunigung ($a_k < 0$) bis zum Stillstand ($v_k = 0$)
4. **Halt / Stillstand**: $a_k = 0$

In [ ]:
# Beispielhafter Beschleunigungsvektor: jeder Wert entspricht der Beschleunigung zum Zeitpunkt t_k
# Dauer je Schritt ist Delta t = 1.0 s
# Sie können dieses Profil beliebig anpassen oder eigene Beschleunigungswerte testen!
a_k_profile = np.array([
    0.0,        # t = 0 s: Start im Stillstand
    1.0, 1.0,   # t = 1..2 s: Beschleunigungsphase (1.0 m/s^2)
    0.5, 0.5,   # t = 3..4 s: Sanftes Weiterbeschleunigen (0.5 m/s^2)
    0.0, 0.0,   # t = 5..6 s: Konstante Fahrt ohne Beschleunigung
    0.0, 0.0,   # t = 7..8 s: Weiterhin gleichförmig
    -1.0, -1.0, # t = 9..10 s: Bremsphase (-1.0 m/s^2)
    -1.0,       # t = 11 s: Letzter Bremsschritt zum Stillstand
    0.0, 0.0    # t = 12..13 s: Fahrrad steht
], dtype=float)

print(f"Anzahl der Zeitschritte: {len(a_k_profile)}")
print(f"Simulationsdauer: {len(a_k_profile) * dt:.1f} s")


## 4. Simulation des Systems mit $x_{k+1} = A \cdot x_k$

Wir berechnen nun schrittweise den zeitlichen Verlauf des Zustandsvektors:
$$
x_k = \begin{bmatrix} p_k \\ v_k \\ a_k \end{bmatrix}
$$

Zum Zeitpunkt $t_k$ wird die Beschleunigung $a_k$ auf den vorgegebenen Vektorwert gesetzt. Anschließend wird der Folgezustand $x_{k+1}$ durch Multiplikation mit der Systemmatrix $A$ berechnet:
$$
x_{k+1} = A \cdot x_k
$$


In [ ]:
def simulate_1d_system(A: np.ndarray, a_profile: np.ndarray, dt: float = 1.0, p_0: float = 0.0, v_0: float = 0.0):
    """Simuliert das 1D lineare kinematische System über die Zeit.
    
    Args:
        A: 3x3 Systemmatrix.
        a_profile: 1D-Array mit diskreten Beschleunigungswerten für jeden Zeitschritt k.
        dt: Zeitschrittweite in Sekunden (Standard: 1.0 s).
        p_0: Anfangsposition in Metern.
        v_0: Anfangsgeschwindigkeit in m/s.
        
    Returns:
        tuple: (time_steps, p_k, v_k, a_k)
    """
    num_steps = len(a_profile)
    
    # Historie der Zustände initialisieren (inklusive Endzustand nach num_steps)
    times = np.zeros(num_steps + 1)
    p_k = np.zeros(num_steps + 1)
    v_k = np.zeros(num_steps + 1)
    a_k = np.zeros(num_steps + 1)
    
    # Anfangszustand bei t = 0
    current_state = np.array([p_0, v_0, a_profile[0]], dtype=float)
    p_k[0] = current_state[0]
    v_k[0] = current_state[1]
    a_k[0] = current_state[2]
    times[0] = 0.0
    
    # Simulationsschleife
    for k in range(num_steps):
        # Aktuelle Beschleunigung zum Zeitschritt k zuweisen
        current_state[2] = a_profile[k]
        a_k[k] = current_state[2]
        
        # Zustandstransition über die Systemmatrix A: x_{k+1} = A * x_k
        next_state = A @ current_state
        
        # Position und Geschwindigkeit für k+1 übernehmen
        p_k[k + 1] = next_state[0]
        v_k[k + 1] = next_state[1]
        
        # Beschleunigung für Zeitschritt k+1 (sofern innerhalb des Profils)
        if k + 1 < num_steps:
            current_state = np.array([next_state[0], next_state[1], a_profile[k + 1]], dtype=float)
            a_k[k + 1] = a_profile[k + 1]
        else:
            current_state = np.array([next_state[0], next_state[1], 0.0], dtype=float)
            a_k[k + 1] = 0.0
            
        times[k + 1] = (k + 1) * dt
        
    return times, p_k, v_k, a_k

# Simulation ausführen
times, p_k, v_k, a_k = simulate_1d_system(A, a_k_profile, dt=dt, p_0=0.0, v_0=0.0)

print(f"Endposition:       p_end = {p_k[-1]:.2f} m")
print(f"Endgeschwindigkeit: v_end = {v_k[-1]:.2f} m/s")
print(f"Maximale Geschw.:  v_max = {np.max(v_k):.2f} m/s ({np.max(v_k) * 3.6:.1f} km/h)")


## 5. Zeitverläufe von Position $p_k$, Geschwindigkeit $v_k$ und Beschleunigung $a_k$

Wir plotten nun die sich ergebende Position, Geschwindigkeit und Beschleunigung über der Zeit.

In [ ]:
# Zeitverläufe mit Hilfsfunktion plotten
fig, axs = helper_module.plot_simulation_results(times, p_k, v_k, a_k_profile)
plt.show()

## 6. Fahrrad-Animation in der 1D-Umgebung

Unter dem Plot visualisieren wir nun das **Fahrrad**, welches mit der simulierten Position $p_k$ und Geschwindigkeit $v_k$ von **links nach rechts** über die 1D-Fahrbahn fährt.

Das Fahrrad wird detailgetreu mit Rahmen, rotierenden Laufrädern, Lenker und Sattel animiert. Die Position des Fahrrads auf der X-Achse entspricht exakt der berechneten Position $p_k$.

In [ ]:
# Fahrrad-Animation mit Hilfsfunktion erstellen
anim = helper_module.create_bicycle_animation(times, p_k, v_k)

# Animation im Notebook als interaktives JavaScript-HTML anzeigen
HTML(anim.to_jshtml())

## 7. Szenario: Beschleunigungsmessung mit Messrauschen (Inertialmesssystem / IMU)

In realen Cyber-Physical Systems (CPS) ist die tatsächliche Beschleunigung $a_k$ nicht exakt bekannt, sondern wird über Sensoren gemessen (beispielsweise über ein **Inertialmesssystem / IMU** am Rad oder Rahmen).

Jede Sensormessung unterliegt jedoch physikalischen Störeinflüssen und Rauschen. Wir modellieren die **gemessene Beschleunigung** $a_{\mathrm{meas}, k}$ als die tatsächliche Beschleunigung $a_k$ überlagert mit normalverteiltem Messrauschen:

$$
a_{\mathrm{meas}, k} = a_k + w_{a, k}, \quad w_{a, k} \sim \mathcal{N}(0, \sigma_a^2)
$$

Wir speisen nun diese gemessene, verrauschte Beschleunigung als Eingangsgröße in unser lineares Systemmodell $x_{k+1} = A \cdot x_k$ ein, um daraus die Geschwindigkeit $v_{\mathrm{calc}}$ und Position $p_{\mathrm{calc}}$ numerisch zu berechnen (Dead Reckoning / Koppelnavigation).

Zusätzlich führen wir eine direkte **gemessene Position** ein, wie sie beispielsweise von einem GPS-Empfänger, Radarsensor oder einer optischen Landmarkenerkennung geliefert wird:

$$
p_{\mathrm{meas}, k} = p_k + v_{p, k}, \quad v_{p, k} \sim \mathcal{N}(0, \sigma_p^2)
$$

In [ ]:
# Reproduzierbarkeit des Rauschens sicherstellen
np.random.seed(42)

# Standardabweichungen des Messrauschens definieren
sigma_a = 0.2   # Messrauschen der Beschleunigung [m/s^2]
sigma_p = 1.0   # Messrauschen der Positionsmessung [m]

# 1. Gemessene Beschleunigung: Ground Truth + Rauschen
a_meas_profile = a_k_profile + np.random.normal(0.0, sigma_a, size=len(a_k_profile))

# 2. Berechnete Position und Geschwindigkeit aus gemessener Beschleunigung über Systemmatrix A
times_calc, p_calc, v_calc, _ = simulate_1d_system(A, a_meas_profile, dt=dt, p_0=0.0, v_0=0.0)

# 3. Gemessene Position: Ground Truth + Rauschen
p_meas = p_k + np.random.normal(0.0, sigma_p, size=len(p_k))

# 4. Gemeinsames Diagramm: Ground Truth vs. berechnete Zustände vs. Messungen
fig, axs = helper_module.plot_comparison_with_measurements(
    times=times,
    p_true=p_k,
    v_true=v_k,
    a_true=a_k_profile,
    p_calc=p_calc,
    v_calc=v_calc,
    a_meas=a_meas_profile,
    p_meas=p_meas
)
plt.show()

## 8. Fragen und Aufgaben zur Analyse

Beantworten Sie anhand des obigen Vergleichsdiagramms folgende Fragen:

### Frage 1: Abweichung der berechneten Position
**Wie verändert sich aus gemessener Beschleunigung die berechnete Position im Vergleich zur tatsächlichen Position?**

*Ihre Antwort:*

---

### Frage 2: Varianzverhalten über der Zeit
**Wie verhält sich die Varianz der berechneten Position im Vergleich zur Varianz der Beschleunigung?**

*Ihre Antwort:*

---

### Frage 3: Vertrauenswürdigkeit von Messung vs. Berechnung
**Wem trauen Sie mehr: der gemessenen Position oder der berechneten Position? Warum?**

*Ihre Antwort:*


## 9. Zusammenfassung & Ausblick

- **Lineares Zustandsraummodell**: Beschreibt die deterministische Kinematik von Position, Geschwindigkeit und Beschleunigung über $x_{k+1} = A \cdot x_k$.
- **Drift bei reiner Integration**: Das Integrieren verrauschter Beschleunigungswerte (Inertialnavigation) führt unvermeidlich zu einer quadratisch/kubisch wachsenden Fehlervarianz und Drift der Position.
- **Messungen vs. Modell**: Direkte Positionsmessungen schwanken um den wahren Wert, driften aber im Gegensatz zur Koppelnavigation nicht weg.
- **Motivation für Kalman-Filter**: Um die Drift der Beschleunigungsintegration zu korrigieren und gleichzeitig das Rauschen der Positionsmessung zu filtern, wird in den nachfolgenden Lerneinheiten der **Kalman-Filter** eingesetzt.